# HydroSeason Global CHIRPS/ERA5 Inland Stress Test

Generate 100 small inland AOIs around the world, fetch monthly rainfall for each AOI with the global product policy, run HydroSeason, and write per-site outputs plus a combined summary table.

The default is CHIRPS v3 monthly rainfall with ERA5 configured as backup. The expensive step is gated by `RUN_FETCH_AND_CLASSIFY = False` so this notebook is safe to open and inspect. Set it to `True` only when you are ready to stream remote rainfall data.


## Setup

Run the install cell if your active notebook kernel does not already have the local checkout installed with the fetch extras.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
# Optional: install this checkout with geospatial fetch dependencies.
# %pip install -e "..[fetch,plot]" -q


In [ ]:
from __future__ import annotations

import json
import time
import traceback
import urllib.request
import zipfile
from dataclasses import asdict

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point, box

from hydroseason import classify_rainfall, get_monthly_aoi_rainfall

try:
    import plotly.express as px
except ImportError:
    px = None


## Configuration

`FETCH_SOURCE = "chirps"` exercises the global product path directly: CHIRPS v3 monthly rainfall first, ERA5 backup when needed. CHIRPS v3 global coverage is approximately 60S-60N, so `MAX_ABS_LATITUDE = 58` keeps the default sample inside CHIRPS coverage. Increase the latitude limit if you deliberately want to test ERA5 fallback outside CHIRPS coverage.

`AOI_SIDE_DEGREES = 0.5` makes each sample a small box suitable for both CHIRPS and ERA5. This is no longer tied to ERA5 grid-cell counts.


In [ ]:
N_SITES = 100
SEED = 20260604

START_YEAR = 1985
END_YEAR = 2023
FETCH_SOURCE = "chirps"  # CHIRPS-first global path. Use "auto" to prefer SILO in Australia.
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
ERA5_FALLBACK = True

AOI_SIDE_DEGREES = 0.5
INLAND_BUFFER_KM = 75
MAX_ABS_LATITUDE = 58

SPATIAL_CHUNK = "auto"
TIME_CHUNK = "auto"
SHOW_FETCH_PROGRESS = True
RAISE_ON_VALIDATION_ERROR = False

RUN_FETCH_AND_CLASSIFY = False
MAX_FAILURES = None

OUTPUT = ROOT / "output" / "global_chirps_era5_stress_test"
CACHE_DIR = ROOT / "data" / "global_chirps_era5_stress_cache"
NATURAL_EARTH_DIR = ROOT / "data" / "natural_earth"

OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NATURAL_EARTH_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT


## Load Global Land Polygons

The sampler uses Natural Earth country polygons, excludes Antarctica, buffers land inward in a projected CRS, and then samples from the remaining inland area. The inward buffer is intentionally larger than the AOI half-diagonal so the generated boxes should avoid sea even before clipping.

In [ ]:
NATURAL_EARTH_COUNTRIES_URL = (
    "https://naturalearth.s3.amazonaws.com/110m_cultural/"
    "ne_110m_admin_0_countries.zip"
)
NATURAL_EARTH_ZIP = NATURAL_EARTH_DIR / "ne_110m_admin_0_countries.zip"
NATURAL_EARTH_EXTRACT_DIR = NATURAL_EARTH_DIR / "ne_110m_admin_0_countries"
NATURAL_EARTH_SHP = NATURAL_EARTH_EXTRACT_DIR / "ne_110m_admin_0_countries.shp"

if not NATURAL_EARTH_ZIP.exists():
    print(f"Downloading {NATURAL_EARTH_COUNTRIES_URL}")
    urllib.request.urlretrieve(NATURAL_EARTH_COUNTRIES_URL, NATURAL_EARTH_ZIP)

if not NATURAL_EARTH_SHP.exists():
    NATURAL_EARTH_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(NATURAL_EARTH_ZIP) as zf:
        zf.extractall(NATURAL_EARTH_EXTRACT_DIR)

land = gpd.read_file(NATURAL_EARTH_SHP).to_crs("EPSG:4326")

name_col = "ADMIN" if "ADMIN" in land.columns else "NAME"
continent_col = "CONTINENT" if "CONTINENT" in land.columns else None

if name_col in land.columns:
    land = land[land[name_col].ne("Antarctica")].copy()

land = land[land.geometry.notna() & ~land.geometry.is_empty].copy()

try:
    land["geometry"] = land.geometry.make_valid()
except AttributeError:
    land["geometry"] = land.geometry.buffer(0)

land = land[land.geometry.notna() & ~land.geometry.is_empty].copy()
land[[name_col] + ([continent_col] if continent_col else [])].head()

## Sample Inland Sites

Sampling is area-weighted, then selected round-robin across continent and latitude-band strata. This is a lightweight way to push HydroSeason through tropical, subtropical, temperate, boreal, and dry inland settings without bringing in a separate climate-classification raster.

In [ ]:
SAMPLE_CRS = "EPSG:8857"  # Equal Earth, metres.


def latitude_band(lat: float) -> str:
    abs_lat = abs(float(lat))
    hemi = "N" if lat >= 0 else "S"
    if abs_lat < 10:
        return "equatorial"
    if abs_lat < 23.5:
        return f"tropical_{hemi}"
    if abs_lat < 35:
        return f"subtropical_{hemi}"
    if abs_lat < 55:
        return f"temperate_{hemi}"
    return f"boreal_subpolar_{hemi}"


def random_point_in_geometry(geom, rng: np.random.Generator, max_attempts: int = 1000):
    minx, miny, maxx, maxy = geom.bounds
    for _ in range(max_attempts):
        candidate = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))
        if geom.contains(candidate):
            return candidate
    return geom.representative_point()


def prepare_inland_polygons(land_gdf: gpd.GeoDataFrame, inland_buffer_km: float):
    work = land_gdf.to_crs(SAMPLE_CRS).copy()
    work = work.explode(ignore_index=True)
    work["geometry"] = work.geometry.buffer(-float(inland_buffer_km) * 1000.0)
    work = work[work.geometry.notna() & ~work.geometry.is_empty].copy()
    work["area_m2"] = work.geometry.area
    work = work[work["area_m2"] > 0].copy()
    return work


def build_candidate_pool(
    inland_gdf: gpd.GeoDataFrame,
    *,
    n_candidates: int,
    seed: int,
) -> gpd.GeoDataFrame:
    rng = np.random.default_rng(seed)
    weights = inland_gdf["area_m2"].to_numpy(dtype=float)
    weights = weights / weights.sum()
    chosen_positions = rng.choice(len(inland_gdf), size=n_candidates, replace=True, p=weights)

    records = []
    geoms = []
    for pos in chosen_positions:
        row = inland_gdf.iloc[int(pos)]
        geoms.append(random_point_in_geometry(row.geometry, rng))
        records.append(
            {
                "country": row[name_col] if name_col in inland_gdf.columns else None,
                "continent": row[continent_col] if continent_col in inland_gdf.columns else "unknown",
            }
        )

    candidates = gpd.GeoDataFrame(records, geometry=geoms, crs=SAMPLE_CRS).to_crs("EPSG:4326")
    candidates["lon"] = candidates.geometry.x
    candidates["lat"] = candidates.geometry.y
    half_side = AOI_SIDE_DEGREES / 2.0
    candidates = candidates[
        candidates["lat"].between(-MAX_ABS_LATITUDE + half_side, MAX_ABS_LATITUDE - half_side)
        & candidates["lon"].between(-180 + half_side, 180 - half_side)
    ].copy()
    candidates["lat_band"] = candidates["lat"].map(latitude_band)
    candidates["stratum"] = candidates["continent"].astype(str) + " / " + candidates["lat_band"]
    return candidates.reset_index(drop=True)


def stratified_select(candidates: gpd.GeoDataFrame, *, n_sites: int, seed: int) -> gpd.GeoDataFrame:
    rng = np.random.default_rng(seed)
    shuffled = candidates.sample(frac=1.0, random_state=seed).copy()
    groups = [list(group.index) for _name, group in shuffled.groupby("stratum", sort=True)]
    rng.shuffle(groups)

    selected = []
    while len(selected) < n_sites:
        progressed = False
        for group in groups:
            if group:
                selected.append(group.pop())
                progressed = True
                if len(selected) == n_sites:
                    break
        if not progressed:
            break

    if len(selected) < n_sites:
        remaining = shuffled.index.difference(selected)
        selected.extend(list(remaining[: n_sites - len(selected)]))

    sites = shuffled.loc[selected].copy().reset_index(drop=True)
    sites["site_id"] = [f"site_{i:03d}" for i in range(1, len(sites) + 1)]
    return sites


inland = prepare_inland_polygons(land, INLAND_BUFFER_KM)
candidates = build_candidate_pool(
    inland,
    n_candidates=max(5000, N_SITES * 80),
    seed=SEED,
)
sites = stratified_select(candidates, n_sites=N_SITES, seed=SEED)

sites[["site_id", "country", "continent", "lat_band", "lon", "lat"]].head()

## Build Small AOIs

Each sampled point becomes a square polygon in EPSG:4326. The default side length is 0.5 degrees, which keeps the stress-test AOIs small while avoiding assumptions about any single product grid.


In [ ]:
def make_aoi_square(lon: float, lat: float, side_degrees: float):
    half = float(side_degrees) / 2.0
    return box(float(lon) - half, float(lat) - half, float(lon) + half, float(lat) + half)


aoi_geometry = [
    make_aoi_square(row.lon, row.lat, AOI_SIDE_DEGREES)
    for row in sites.itertuples(index=False)
]
site_aois = gpd.GeoDataFrame(
    sites.drop(columns="geometry"),
    geometry=aoi_geometry,
    crs="EPSG:4326",
)
site_aois["aoi_side_degrees"] = AOI_SIDE_DEGREES
site_aois["inland_buffer_km"] = INLAND_BUFFER_KM

sites_csv = OUTPUT / "global_chirps_era5_stress_sites.csv"
sites_geojson = OUTPUT / "global_chirps_era5_stress_sites.geojson"

site_aois.drop(columns="geometry").to_csv(sites_csv, index=False)
site_aois.to_file(sites_geojson, driver="GeoJSON")

print(f"Wrote {sites_csv}")
print(f"Wrote {sites_geojson}")
site_aois[["site_id", "country", "continent", "lat_band", "lon", "lat"]].head()

In [ ]:
site_aois.groupby(["continent", "lat_band"]).size().rename("n_sites").reset_index().sort_values(
    ["continent", "lat_band"]
)

In [ ]:
if px is not None:
    fig = px.scatter_geo(
        site_aois,
        lon="lon",
        lat="lat",
        color="lat_band",
        hover_name="site_id",
        hover_data=["country", "continent"],
        title="Sampled inland CHIRPS/ERA5 stress-test AOIs",
    )
    fig.update_geos(showland=True, landcolor="rgb(235, 235, 225)")
    fig.show()
else:
    print("Install plotly to view the sample map.")


## Fetch CHIRPS/ERA5 And Run HydroSeason

This is the expensive cell. It writes partial progress after every site, so you can stop and resume without losing the summary rows already completed. The cache is shared across all sites. Each monthly CSV keeps `Data_Source`, `Data_Product`, and `Fetch_Note`, so mixed CHIRPS/ERA5 fallback series remain visible.


In [ ]:
def assert_pipeline_invariants(artifacts) -> None:
    result = artifacts.result
    required_columns = {"Date", "Year", "Month", "Rainfall_mm", "SeasonType", "Hydro_Year"}
    missing_columns = required_columns.difference(result.columns)
    if missing_columns:
        raise AssertionError(f"Missing result columns: {sorted(missing_columns)}")
    if result.empty:
        raise AssertionError("Pipeline returned an empty result")

    dates = pd.to_datetime(result["Date"])
    if not dates.is_monotonic_increasing:
        raise AssertionError("Result dates are not sorted")
    if not result["Month"].astype(int).equals(dates.dt.month.astype(int)):
        raise AssertionError("Month column does not match Date")

    allowed_seasons = {"Wet", "Dry", "Unclassified"}
    observed_seasons = set(result["SeasonType"].astype(str).unique())
    unexpected = observed_seasons.difference(allowed_seasons)
    if unexpected:
        raise AssertionError(f"Unexpected SeasonType values: {sorted(unexpected)}")

    allowed_regimes = {"seasonal", "borderline", "non_seasonal"}
    if artifacts.diagnostics.regime not in allowed_regimes:
        raise AssertionError(f"Unexpected diagnostics regime: {artifacts.diagnostics.regime}")
    if not np.isfinite(artifacts.diagnostics.stl_strength):
        raise AssertionError("STL strength is not finite")


def site_metadata(row) -> dict:
    return {
        "site_id": row.site_id,
        "country": row.country,
        "continent": row.continent,
        "lat_band": row.lat_band,
        "lon": float(row.lon),
        "lat": float(row.lat),
        "aoi_side_degrees": float(row.aoi_side_degrees),
        "inland_buffer_km": float(row.inland_buffer_km),
    }


def run_site(row) -> dict:
    metadata = site_metadata(row)
    site_id = metadata["site_id"]
    site_dir = OUTPUT / site_id
    site_dir.mkdir(parents=True, exist_ok=True)

    site_gdf = gpd.GeoDataFrame([metadata], geometry=[row.geometry], crs="EPSG:4326")
    site_gdf.to_file(site_dir / f"{site_id}_aoi.geojson", driver="GeoJSON")

    started = time.perf_counter()
    try:
        monthly = get_monthly_aoi_rainfall(
            site_gdf,
            start_year=START_YEAR,
            end_year=END_YEAR,
            source=FETCH_SOURCE,
            era5_zarr_path=ERA5_ZARR,
            cache_dir=CACHE_DIR,
            spatial_chunk=SPATIAL_CHUNK,
            time_chunk=TIME_CHUNK,
            era5_fallback=ERA5_FALLBACK,
            show_progress=SHOW_FETCH_PROGRESS,
        )
        monthly.to_csv(site_dir / f"{site_id}_monthly_rainfall.csv", index=False)

        artifacts = classify_rainfall(
            monthly,
            raise_on_validation_error=RAISE_ON_VALIDATION_ERROR,
        )
        assert_pipeline_invariants(artifacts)

        artifacts.result.to_csv(site_dir / f"{site_id}_hydroseason_result.csv", index=False)
        artifacts.fixed_monthly.to_csv(site_dir / f"{site_id}_fixed_monthly.csv", index=False)
        if artifacts.wet_boundaries is not None:
            artifacts.wet_boundaries.to_csv(site_dir / f"{site_id}_wet_boundaries.csv", index=False)

        diagnostics = asdict(artifacts.diagnostics)
        (site_dir / f"{site_id}_diagnostics.json").write_text(
            json.dumps(diagnostics, default=str, indent=2),
            encoding="utf-8",
        )

        elapsed_seconds = time.perf_counter() - started
        return {
            **metadata,
            "status": "ok",
            "elapsed_seconds": elapsed_seconds,
            "rows_monthly": int(len(monthly)),
            "rows_result": int(len(artifacts.result)),
            "data_sources": ",".join(sorted(monthly.get("Data_Source", pd.Series(dtype=str)).dropna().astype(str).unique())),
            "regime": diagnostics["regime"],
            "regime_source": diagnostics["regime_source"],
            "stl_strength": diagnostics["stl_strength"],
            "walsh_lawler_si": diagnostics["walsh_lawler_si"],
            "hydro_year_start_month": diagnostics["hydro_year_start_month"],
            "data_confidence": diagnostics["data_confidence"],
            "n_imputed": diagnostics["n_imputed"],
            "max_consecutive_missing": diagnostics["max_consecutive_missing"],
            "error": None,
        }
    except Exception as exc:  # noqa: BLE001 - keep stress-test failures inspectable.
        elapsed_seconds = time.perf_counter() - started
        error_text = traceback.format_exc()
        (site_dir / f"{site_id}_error.txt").write_text(error_text, encoding="utf-8")
        return {
            **metadata,
            "status": "failed",
            "elapsed_seconds": elapsed_seconds,
            "rows_monthly": None,
            "rows_result": None,
            "data_sources": None,
            "regime": None,
            "regime_source": None,
            "stl_strength": None,
            "walsh_lawler_si": None,
            "hydro_year_start_month": None,
            "data_confidence": None,
            "n_imputed": None,
            "max_consecutive_missing": None,
            "error": repr(exc),
        }

In [ ]:
summary_path = OUTPUT / "global_chirps_era5_stress_summary.csv"

if RUN_FETCH_AND_CLASSIFY:
    summaries = []
    failed_count = 0

    for idx, row in enumerate(site_aois.itertuples(index=False), start=1):
        print(f"[{idx:03d}/{len(site_aois):03d}] {row.site_id} {row.country} ({row.lat:.3f}, {row.lon:.3f})")
        summary = run_site(row)
        summaries.append(summary)

        summary_df = pd.DataFrame(summaries)
        summary_df.to_csv(summary_path, index=False)

        if summary["status"] != "ok":
            failed_count += 1
            print(f"  failed: {summary['error']}")
            if MAX_FAILURES is not None and failed_count >= MAX_FAILURES:
                print(f"Stopping after {failed_count} failures because MAX_FAILURES={MAX_FAILURES}.")
                break
        else:
            print(
                "  ok "
                f"sources={summary['data_sources']} "
                f"regime={summary['regime']} "
                f"SI={summary['walsh_lawler_si']:.3f} "
                f"STL={summary['stl_strength']:.3f} "
                f"seconds={summary['elapsed_seconds']:.1f}"
            )

    display(pd.DataFrame(summaries))
else:
    print(f"Set RUN_FETCH_AND_CLASSIFY = True when you are ready to fetch {FETCH_SOURCE}/ERA5 backup rainfall and run HydroSeason.")
    print(f"Planned sites: {len(site_aois)}")
    print(f"Summary will be written to: {summary_path}")


## Inspect Results

Run these cells after the stress test has produced `global_chirps_era5_stress_summary.csv`.


In [ ]:
if summary_path.exists():
    stress_summary = pd.read_csv(summary_path)
    display(stress_summary.head())
    display(stress_summary["status"].value_counts(dropna=False).rename("n_sites"))
else:
    print(f"No summary file yet: {summary_path}")

In [ ]:
if summary_path.exists():
    stress_summary = pd.read_csv(summary_path)
    ok = stress_summary[stress_summary["status"].eq("ok")].copy()
    display(ok.groupby(["continent", "lat_band", "data_sources", "regime"]).size().rename("n_sites").reset_index())
    display(
        ok[["elapsed_seconds", "stl_strength", "walsh_lawler_si", "n_imputed", "max_consecutive_missing"]]
        .describe(percentiles=[0.5, 0.9, 0.95])
    )
else:
    print("Run the stress test first.")

In [ ]:
if summary_path.exists() and px is not None:
    stress_summary = pd.read_csv(summary_path)
    fig = px.scatter_geo(
        stress_summary,
        lon="lon",
        lat="lat",
        color="regime",
        symbol="status",
        hover_name="site_id",
        hover_data=["country", "continent", "lat_band", "data_sources", "walsh_lawler_si", "stl_strength", "error"],
        title="HydroSeason CHIRPS/ERA5 stress-test outcomes",
    )
    fig.update_geos(showland=True, landcolor="rgb(235, 235, 225)")
    fig.show()
else:
    print("Run the stress test and install plotly to view the outcome map.")

## Optional: Generate A Report For One Site

Set `REPORT_SITE_ID` to a completed site, for example `"site_001"`, to regenerate a single-site HTML report from its cached monthly CSV.


In [ ]:
REPORT_SITE_ID = None

if REPORT_SITE_ID:
    from hydroseason import generate_html_report

    site_dir = OUTPUT / REPORT_SITE_ID
    monthly_path = site_dir / f"{REPORT_SITE_ID}_monthly_rainfall.csv"
    monthly = pd.read_csv(monthly_path)
    artifacts = classify_rainfall(monthly, raise_on_validation_error=RAISE_ON_VALIDATION_ERROR)
    report_path = generate_html_report(
        artifacts,
        site_dir / f"{REPORT_SITE_ID}_hydroseason_report.html",
        title=f"HydroSeason CHIRPS/ERA5 stress test - {REPORT_SITE_ID}",
    )
    report_path
else:
    print("Set REPORT_SITE_ID to generate a single-site report.")
